# nflverse NFL Schedule Ingestion (2016-2025)

Pull **complete NFL game schedules** from nflverse (open source, free, comprehensive).

## 🎯 Target Coverage

**Regular Season Games by Era:**

| Season Range | Weeks | Games/Team | Total League Games |
|--------------|-------|------------|--------------------|
| 2016-2020    | 17    | 16         | **256/season**     |
| 2021-2025    | 18    | 17         | **272/season**     |

**Total Target:** ~2,640 regular season games (2016-2025)

**Week 18 Context (2021+):**
- Not an "extra" game for every team
- Final week of 18-week schedule (17 games + 1 bye)
- Fantasy leagues typically end Week 17 (playoff-bound teams may rest starters Week 18)

---

## 🔄 Two-Phase Ingestion Pattern

1. **HISTORICAL_MODE = True**: Fetch complete 10-year history (2016-2025)
2. **HISTORICAL_MODE = False**: Fetch current season only (incremental updates)

Switch to `False` after initial historical load is validated.

---

## 📊 Data Source

**nflverse** - Open source NFL data aggregation project
- **Website**: https://nflverse.nflverse.com
- **GitHub**: https://github.com/nflverse
- **Coverage**: Complete schedules 1999-present
- **Cost**: FREE (open source)
- **API**: Python package `nfl_data_py`

In [0]:
# Install nflverse Python package
%pip install nfl_data_py --quiet

In [0]:
import nfl_data_py as nfl
import pandas as pd
import json
from pyspark.sql import Row
from pyspark.sql import functions as F
from datetime import datetime

# ============================================================================
# CONFIGURATION MODE
# ============================================================================
# Set to True to fetch ALL historical schedules (2016-2025)
# Set to False to fetch only LATEST season (current year)
HISTORICAL_MODE = False  # ✅ Switched to incremental updates
# ============================================================================

if HISTORICAL_MODE:
    # Fetch 10 years of schedules (2016-2025)
    SCHEDULE_SEASONS = list(range(2016, 2026))  # 2016 through 2025
    print(f"🔄 HISTORICAL MODE: Fetching schedules for {len(SCHEDULE_SEASONS)} seasons ({SCHEDULE_SEASONS[0]}-{SCHEDULE_SEASONS[-1]})")
    print(f"   Expected: ~256 games/season (2016-2020), ~272 games/season (2021-2025)")
    print(f"   Total target: ~2,640 regular season games")
else:
    # Fetch only current season
    current_year = datetime.now().year
    SCHEDULE_SEASONS = [current_year]
    print(f"⚡ LATEST MODE: Fetching schedule for current season ({SCHEDULE_SEASONS[0]})")
    print(f"   This will fetch ~272 games (regular season + playoffs) for incremental updates")

print(f"\n📊 Schedule Seasons: {SCHEDULE_SEASONS}")

In [0]:
# Fetch NFL game schedules using nflverse
print("\n" + "=" * 80)
print("🏈 Fetching NFL Game Schedules from nflverse")
print("=" * 80)

all_schedules = []
schedule_stats = {}

for season in SCHEDULE_SEASONS:
    try:
        print(f"\n📅 Season {season}: Fetching...", end=" ")
        
        # Import schedule data for the season
        schedule_df = nfl.import_schedules([season])
        
        if schedule_df is not None and len(schedule_df) > 0:
            game_count = len(schedule_df)
            schedule_stats[season] = game_count
            
            # Add season column if not present
            if 'season' not in schedule_df.columns:
                schedule_df['season'] = season
            
            all_schedules.append(schedule_df)
            print(f"✅ {game_count} games")
        else:
            schedule_stats[season] = 0
            print(f"⚠️ 0 games")
            
    except Exception as e:
        print(f"❌ Error: {str(e)[:60]}")
        schedule_stats[season] = 0

print("\n" + "=" * 80)
print("📊 Summary by Season")
print("=" * 80)

total_games = 0
for season in sorted(schedule_stats.keys()):
    count = schedule_stats[season]
    total_games += count
    era = "(17 weeks, 16 games/team)" if season <= 2020 else "(18 weeks, 17 games/team)"
    print(f"   {season}: {count:3d} games {era}")

print(f"\n   {'TOTAL':>5}: {total_games:4d} games")
print("=" * 80)

if all_schedules:
    # Combine all seasons into single DataFrame
    schedules_combined = pd.concat(all_schedules, ignore_index=True)
    print(f"\n✅ Successfully fetched {len(schedules_combined)} total games from {len(SCHEDULE_SEASONS)} season(s)")
    print(f"\n📊 Available columns ({len(schedules_combined.columns)} total):")
    print(f"   {list(schedules_combined.columns)[:20]}")
    print(f"   ...")
    print(f"\n📊 Sample games:")
    display(schedules_combined[['season', 'week', 'gameday', 'weekday', 'gametime', 'home_team', 'away_team', 'home_score', 'away_score', 'stadium']].head(10))
else:
    print("\n❌ No schedules fetched")
    schedules_combined = pd.DataFrame()

In [0]:
# Check what game types are included
if 'schedules_combined' in locals() and len(schedules_combined) > 0:
    print("\n📊 Game Type Distribution")
    print("=" * 80)
    
    if 'game_type' in schedules_combined.columns:
        game_type_counts = schedules_combined.groupby(['season', 'game_type']).size().reset_index(name='count')
        print("\nGames by Season and Type:")
        for season in sorted(schedules_combined['season'].unique()):
            season_data = game_type_counts[game_type_counts['season'] == season]
            print(f"\n  {season}:")
            for _, row in season_data.iterrows():
                print(f"    {row['game_type']:10s} {row['count']:3d} games")
    else:
        print("\n⚠️ 'game_type' column not available in data")
    
    # Show week distribution
    print("\n\n📊 Week Distribution (Regular Season)")
    print("=" * 80)
    
    # Filter for regular season only
    if 'game_type' in schedules_combined.columns:
        reg_season = schedules_combined[schedules_combined['game_type'] == 'REG']
    else:
        # If no game_type, assume all are regular season
        reg_season = schedules_combined
    
    print(f"\nRegular season games: {len(reg_season)}")
    print(f"Weeks covered: {sorted(reg_season['week'].unique())}")
    print(f"\nGames per week (sample from most recent season):")
    
    latest_season = reg_season['season'].max()
    latest_week_counts = reg_season[reg_season['season'] == latest_season].groupby('week').size()
    print(f"\n  Season {latest_season}:")
    for week, count in latest_week_counts.items():
        print(f"    Week {week:2d}: {count:2d} games")
else:
    print("\n⚠️ No schedule data available for analysis")

In [0]:
# Transform schedule data to Delta table schema
if 'schedules_combined' in locals() and len(schedules_combined) > 0:
    print("\n" + "=" * 80)
    print("💾 Transforming Schedule Data to Spark DataFrame")
    print("=" * 80)
    
    rows = []
    for idx, game in schedules_combined.iterrows():
        # Extract game information
        game_id = str(game.get('game_id', ''))
        season = int(game.get('season', 0))
        week = int(game.get('week', 0))
        game_type = str(game.get('game_type', 'REG'))  # REG, POST, PRE
        gameday = str(game.get('gameday', ''))
        gametime = str(game.get('gametime', ''))
        home_team = str(game.get('home_team', ''))
        away_team = str(game.get('away_team', ''))
        home_score = int(game.get('home_score', 0) if pd.notna(game.get('home_score')) else 0)
        away_score = int(game.get('away_score', 0) if pd.notna(game.get('away_score')) else 0)
        stadium = str(game.get('stadium', ''))
        location = str(game.get('location', ''))
        
        # Build event name
        event_name = f"{away_team} vs {home_team}"
        
        # Convert full row to JSON for stats column
        stats_json = json.dumps(game.to_dict(), default=str)
        
        rows.append(
            Row(
                event_id=game_id,
                event_name=event_name,
                date=gameday,
                time=gametime,
                home_team=home_team,
                away_team=away_team,
                home_score=home_score,
                away_score=away_score,
                venue=stadium,
                season=season,
                week=week,
                game_type=game_type,
                source="nflverse",
                stats=stats_json
            )
        )
    
    schedules_spark_df = spark.createDataFrame(rows)
    print(f"\n✅ Created Spark DataFrame with {schedules_spark_df.count()} games")
    
    # Show schema
    print("\n📋 Schema:")
    schedules_spark_df.printSchema()
    
    # Show game type breakdown
    print("\n📊 Games by Type:")
    schedules_spark_df.groupBy("game_type").count().orderBy("game_type").show()
    
    # Show sample
    print("\n📊 Sample transformed data:")
    display(schedules_spark_df.select('season', 'week', 'game_type', 'date', 'home_team', 'away_team', 'venue').limit(10))
    
else:
    print("\n⚠️ No schedule data to transform")

In [0]:
# Write schedules to bronze_nfl_games table
if 'schedules_spark_df' in locals():
    print("\n" + "=" * 80)
    print("💾 Writing Schedules to bronze_nfl_games")
    print("=" * 80)
    
    bronze_schedules = schedules_spark_df.withColumn("ingested_at", F.current_timestamp())
    
    # Write to Delta table (append mode)
    bronze_schedules.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable("main.fantasai.bronze_nfl_games")
    
    total_written = bronze_schedules.count()
    print(f"\n✅ Appended {total_written} games to bronze_nfl_games")
    
    # Show breakdown by season and game type
    print("\n📊 Breakdown by Season & Game Type:")
    breakdown = bronze_schedules.groupBy("season", "game_type").count().orderBy("season", "game_type")
    breakdown.show(50)
    
else:
    print("\n⚠️ No schedules to write")

In [0]:
%sql
-- Compare all sources in bronze_nfl_games
SELECT 
  source,
  season,
  COUNT(*) as game_count,
  MIN(date) as first_game_date,
  MAX(date) as last_game_date
FROM main.fantasai.bronze_nfl_games
GROUP BY source, season
ORDER BY season DESC, source

In [0]:
%sql
-- Detailed breakdown of nflverse schedule data
SELECT 
  season,
  game_type,
  COUNT(*) as game_count,
  MIN(week) as first_week,
  MAX(week) as last_week
FROM main.fantasai.bronze_nfl_games
WHERE source = 'nflverse'
GROUP BY season, game_type
ORDER BY season DESC, game_type

## ✅ Next Steps After Validation

### 1. Switch to Latest-Only Mode

Once historical data is validated:

```python
# In Cell 3 (Configuration), change:
HISTORICAL_MODE = False  # Switch to incremental updates
```

This will fetch only the current season on subsequent runs.

---

### 2. Schedule Regular Updates

Set up a job to run this notebook:
- **During season**: Weekly (after Monday Night Football)
- **Off-season**: Monthly (for schedule updates)

---

### 3. Clean Up Old Ingestion Notebooks

Once nflverse schedule data is validated, consider archiving:
- `17_thesportsdb_ingestion` (only provided 15 preseason games/season)

**nflverse advantages:**
- ✅ Complete regular season schedules (~256-272 games/season)
- ✅ Includes playoffs and preseason
- ✅ Free and open source
- ✅ Actively maintained by NFL data community
- ✅ Same source as player stats (consistent team names/IDs)

---

### 4. Compare Data Quality

**TheSportsDB:**
- 165 games total (2015-2025)
- Only preseason games
- Missing 94% of regular season data

**nflverse:**
- ~2,640 regular season games (2016-2025)
- Plus playoffs and preseason
- Complete coverage ✅